In [1]:
# Section 0: Imports
from pathlib import Path
from collections import Counter, defaultdict
import os
import sys
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

from skimage.color import rgb2gray
from skimage.transform import resize
from skimage.feature import hog

from sklearn.decomposition import PCA
from numpy.linalg import norm

from scipy.stats import skew, kurtosis, normaltest
from scipy.linalg import cho_factor, cho_solve

In [2]:
# Section 0.1: Paths, Config

ROOT = Path("/work/zq63/Bayes/Data/casia_webface/casia-webface")
REC_PATH = ROOT / "train.rec"
IDX_PATH = ROOT / "train.idx"
LST_PATH = ROOT / "train.lst"

CACHE_DIR = Path("/work/zq63/Bayes/Data/cache_casia_recordio")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR = Path("/work/zq63/Bayes/Data/casia_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

m = 5
seed = 0

# runtime caps (recommended)
MAX_TRAIN_IDS = 800
MAX_TRAIN_IMGS_PER_ID = 10
MAX_TEST_IDS = 400
MAX_TEST_IMGS_PER_ID = 10

N_SAME = 500
N_DIFF = 500

H, W = 128, 128
PCA_DIM = 50

random.seed(seed)
np.random.seed(seed)

In [3]:
# Section 0.3: Extract zip (one-time)
"""
def ensure_extracted(zip_path: Path, out_dir: Path) -> None:
    if out_dir.exists() and any(out_dir.iterdir()):
        print(f"[OK] Extract dir already non-empty: {out_dir}")
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Extracting {zip_path} -> {out_dir}")
    # Use system unzip for speed if available
    cmd = f'unzip -q "{zip_path}" -d "{out_dir}"'
    ret = os.system(cmd)
    if ret != 0:
        raise RuntimeError("unzip failed. Check that `unzip` exists and the zip is valid.")
    print("[OK] Extraction done.")

ensure_extracted(DATA_ZIP, EXTRACT_DIR)
"""

'\ndef ensure_extracted(zip_path: Path, out_dir: Path) -> None:\n    if out_dir.exists() and any(out_dir.iterdir()):\n        print(f"[OK] Extract dir already non-empty: {out_dir}")\n        return\n    out_dir.mkdir(parents=True, exist_ok=True)\n    print(f"[INFO] Extracting {zip_path} -> {out_dir}")\n    # Use system unzip for speed if available\n    cmd = f\'unzip -q "{zip_path}" -d "{out_dir}"\'\n    ret = os.system(cmd)\n    if ret != 0:\n        raise RuntimeError("unzip failed. Check that `unzip` exists and the zip is valid.")\n    print("[OK] Extraction done.")\n\nensure_extracted(DATA_ZIP, EXTRACT_DIR)\n'

In [4]:
# Section 0.4: Sanity check files
print("Python:", sys.version)
for p in [REC_PATH, IDX_PATH, LST_PATH]:
    print(p, "exists:", p.exists(), "size(MB):", (p.stat().st_size / 1024 / 1024) if p.exists() else None)

if not (REC_PATH.exists() and IDX_PATH.exists() and LST_PATH.exists()):
    raise FileNotFoundError("Missing required files under ROOT. Check paths.")

Python: 3.9.25 (main, Nov  3 2025, 22:33:05) 
[GCC 11.2.0]
/work/zq63/Bayes/Data/casia_webface/casia-webface/train.rec exists: True size(MB): 2599.757785797119
/work/zq63/Bayes/Data/casia_webface/casia-webface/train.idx exists: True size(MB): 8.304418563842773
/work/zq63/Bayes/Data/casia_webface/casia-webface/train.lst exists: True size(MB): 69.77837085723877


In [5]:
# Section 0.4: Peek train.lst structure 

with open(LST_PATH, "rb") as f:
    b = f.read(256)
print("First bytes repr:", b[:80])

with open(LST_PATH, "rb") as f:
    raw_lines = [f.readline() for _ in range(3)]
print("---- first 3 raw lines (latin1 repr) ----")
for i, rl in enumerate(raw_lines, 1):
    print(i, repr(rl.decode("latin1", errors="replace")))


First bytes repr: b'0\t/raid5data/dplearn/CASIA-WebFace/0000045/001.jpg\t0\t57\t59\t187\t212\t93.3661\t151.5'
---- first 3 raw lines (latin1 repr) ----
1 '0\t/raid5data/dplearn/CASIA-WebFace/0000045/001.jpg\t0\t57\t59\t187\t212\t93.3661\t151.566\t127.419\t107.732\t158.046\t115.133\t105.133\t139.307\t179.872\t170.261\n'
2 '0\t/raid5data/dplearn/CASIA-WebFace/0000045/013.jpg\t0\t55\t50\t168\t199\t97.5698\t152.193\t132.47\t104.22\t147.564\t102.114\t99.1113\t140.006\t169.167\t166.637\n'
3 '0\t/raid5data/dplearn/CASIA-WebFace/0000045/012.jpg\t0\t57\t45\t187\t218\t92.6203\t150.45\t123.761\t106.671\t149.209\t109.181\t103.016\t141.798\t179.75\t174.899\n'


In [6]:
# Section 1: Parse train.lst

def parse_lst_kaggle(lst_path: Path, keep_path: bool = False) -> pd.DataFrame:
    rows = []
    with open(lst_path, "rb") as f:
        for row_i, rb in enumerate(f):
            line = rb.decode("latin1", errors="ignore")
            # remove null chars and whitespace
            line = line.replace("\x00", "").strip()
            if not line:
                continue

            parts = line.split("\t")
            # find a field that looks like an image path
            # (your sample has path in parts[1], but we make it robust)
            path_field = None
            for tok in parts:
                t = tok.strip()
                if ("/" in t) and (t.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))):
                    path_field = t
                    break
            if path_field is None:
                # no path, skip
                continue

            pid = Path(path_field).parent.name  # identity folder
            if keep_path:
                rows.append((row_i, pid, path_field))
            else:
                rows.append((row_i, pid))

    if keep_path:
        return pd.DataFrame(rows, columns=["rec_idx_candidate", "id", "raw_path"])
    return pd.DataFrame(rows, columns=["rec_idx_candidate", "id"])


lst_df = parse_lst_kaggle(LST_PATH, keep_path=False)
print("Rows parsed from train.lst:", lst_df.shape[0])
print(lst_df.head())
print("Unique identities (from path folder):", lst_df["id"].nunique())

Rows parsed from train.lst: 494149
   rec_idx_candidate       id
0                  0  0000045
1                  1  0000045
2                  2  0000045
3                  3  0000045
4                  4  0000045
Unique identities (from path folder): 10572


In [7]:
# Section 2: Identity stats (labels act as identities)

cnt = Counter(lst_df["id"].tolist())
print("Unique identities:", len(cnt))
print("Top 10 identities by #images:", cnt.most_common(10))

sizes = np.array(list(cnt.values()))
print(">=2 images identities:", int((sizes >= 2).sum()))
print(">=3 images identities:", int((sizes >= 3).sum()))
print(">=5 images identities:", int((sizes >= 5).sum()))
print(">=10 images identities:", int((sizes >= 10).sum()))

Unique identities: 10572
Top 10 identities by #images: [('0004266', 804), ('0519456', 745), ('0000439', 711), ('0010736', 653), ('0515116', 648), ('0221043', 636), ('0424060', 629), ('0004770', 624), ('0358316', 617), ('0221046', 611)]
>=2 images identities: 10572
>=3 images identities: 10571
>=5 images identities: 10569
>=10 images identities: 10542


In [8]:
# Section 3: Filter identities with >= m images

keep_ids = {pid for pid, k in cnt.items() if k >= m}
print(f"Identities with >= {m} images:", len(keep_ids))

lst_keep = lst_df[lst_df["id"].isin(keep_ids)].copy()
print("Images kept:", lst_keep.shape[0])


# Build by_id mapping: id -> list of rec_idx_candidate
by_id = defaultdict(list)
for r in lst_keep.itertuples(index=False):
    by_id[r.id].append(int(r.rec_idx_candidate))

print("Min images per kept identity:", min(len(v) for v in by_id.values()))
print("Avg images per kept identity:", float(np.mean([len(v) for v in by_id.values()])))
print("Max images per kept identity:", max(len(v) for v in by_id.values()))

Identities with >= 5 images: 10569
Images kept: 494139
Min images per kept identity: 5
Avg images per kept identity: 46.753619074652285
Max images per kept identity: 804


In [9]:
# Section 4: Open RecordIO + decide key mapping

try:
    import mxnet as mx
    from mxnet import recordio
except Exception as e:
    raise RuntimeError(
        "mxnet import failed. You need mxnet to read train.rec.\n"
        "If pip install fails due to Python version, create a conda env with python=3.9.\n"
        f"Original error: {e}"
    )

rec = recordio.MXIndexedRecordIO(str(IDX_PATH), str(REC_PATH), "r")

def get_all_record_keys(rec_obj):
    # try common internal attributes
    for attr in ["keys", "idx"]:
        if hasattr(rec_obj, attr):
            val = getattr(rec_obj, attr)
            # rec.idx is often a dict-like mapping key->pos
            if isinstance(val, dict):
                return sorted(list(val.keys()))
            # rec.keys could be a list/iterable
            try:
                return sorted(list(val))
            except Exception:
                pass
    # last resort: try rec_obj.idx.keys()
    try:
        return sorted(list(rec_obj.idx.keys()))
    except Exception:
        return None

all_keys = get_all_record_keys(rec)
print("RecordIO keys available:", None if all_keys is None else len(all_keys))

def can_read_key(k: int) -> bool:
    try:
        s = rec.read_idx(int(k))
        return s is not None
    except Exception:
        return False

def choose_key_for_row_mapping(df: pd.DataFrame, all_keys_list):
    """
    Decide how to map row index (rec_idx_candidate) to actual RecordIO key.
    Strategy:
      A) test if key = row_i works (high success rate) -> use row_i
      B) else if keys count matches rows -> map by sorted(keys)[row_order]
      C) else fail loudly
    """
    candidates = df["rec_idx_candidate"].values
    if len(candidates) == 0:
        raise ValueError("No rows in lst_keep; cannot map keys.")

    # sample up to 200 indices to test
    sample_n = min(200, len(candidates))
    sample = np.random.choice(candidates, size=sample_n, replace=False)

    ok = 0
    for k in sample:
        if can_read_key(int(k)):
            ok += 1
    rate = ok / sample_n
    print(f"Test mapping key=row_i success rate: {rate:.3f} ({ok}/{sample_n})")

    if rate >= 0.80:
        print("[MAP] Using key = rec_idx_candidate (row number).")
        return "row_i"

    if all_keys_list is not None:
        if len(all_keys_list) >= df.shape[0]:
            print("[MAP] Using key = sorted(train.idx keys)[row_order].")
            return "sorted_keys"

    raise RuntimeError(
        "Cannot map train.lst rows to RecordIO keys automatically.\n"
        "Need further inspection of train.idx/train.rec indexing."
    )

mapping_mode = choose_key_for_row_mapping(lst_keep, all_keys)

if mapping_mode == "row_i":
    # rec_key is exactly the row number
    lst_keep["rec_key"] = lst_keep["rec_idx_candidate"].astype(np.int64)
elif mapping_mode == "sorted_keys":
    # map row order to sorted keys (assumes 1-1 ordering)
    # IMPORTANT: this assumes lst_keep is in original order; keep it stable
    # We'll rebuild lst_keep in original file order by sorting rec_idx_candidate
    lst_keep = lst_keep.sort_values("rec_idx_candidate").reset_index(drop=True)
    keys_sorted = np.array(all_keys[: lst_keep.shape[0]], dtype=np.int64)
    lst_keep["rec_key"] = keys_sorted
else:
    raise RuntimeError("Unexpected mapping_mode.")

print(lst_keep.head())

RecordIO keys available: 501196
Test mapping key=row_i success rate: 1.000 (200/200)
[MAP] Using key = rec_idx_candidate (row number).
   rec_idx_candidate       id  rec_key
0                  0  0000045        0
1                  1  0000045        1
2                  2  0000045        2
3                  3  0000045        3
4                  4  0000045        4


In [10]:
# Section 5: Identity split + build train/test pools (same as before)
all_ids = sorted(by_id.keys())
random.shuffle(all_ids)

n_train = int(0.7 * len(all_ids))
train_ids_all = all_ids[:n_train]
test_ids_all = all_ids[n_train:]

train_ids = train_ids_all[:MAX_TRAIN_IDS]
test_ids = test_ids_all[:MAX_TEST_IDS]

print("Train identities:", len(train_ids))
print("Test identities:", len(test_ids))

# Build a fast lookup: rec_idx_candidate -> rec_key
cand_to_key = dict(zip(lst_keep["rec_idx_candidate"].astype(int).tolist(),
                       lst_keep["rec_key"].astype(int).tolist()))

# TRAIN image keys
train_keys = []
train_labels = []
for pid in train_ids:
    cand_list = by_id[pid]
    random.shuffle(cand_list)
    cand_list = cand_list[:MAX_TRAIN_IMGS_PER_ID]
    for cand in cand_list:
        if cand in cand_to_key:
            train_keys.append(int(cand_to_key[cand]))
            train_labels.append(pid)

print("Train images used:", len(train_keys))

# TEST pool keys
test_pool = {}
for pid in test_ids:
    cand_list = by_id[pid]
    random.shuffle(cand_list)
    cand_list = cand_list[:MAX_TEST_IMGS_PER_ID]
    keys = [int(cand_to_key[c]) for c in cand_list if c in cand_to_key]
    if len(keys) > 0:
        test_pool[pid] = keys

print("Test ids with >=1 key:", len(test_pool))


def sample_same_pairs(pool: dict[str, list[int]], n_pairs: int) -> pd.DataFrame:
    valid = [k for k, v in pool.items() if len(v) >= 2]
    if not valid:
        raise ValueError("No test identity has >=2 images in pool.")
    rows = []
    for _ in range(n_pairs):
        pid = random.choice(valid)
        a, b = random.sample(pool[pid], 2)
        rows.append((pid, a, pid, b, 1))
    return pd.DataFrame(rows, columns=["id1", "key1", "id2", "key2", "label"])


def sample_diff_pairs(pool: dict[str, list[int]], n_pairs: int) -> pd.DataFrame:
    valid = [k for k, v in pool.items() if len(v) >= 1]
    if len(valid) < 2:
        raise ValueError("Need at least 2 test identities with >=1 image.")
    rows = []
    for _ in range(n_pairs):
        pid1, pid2 = random.sample(valid, 2)
        a = random.choice(pool[pid1])
        b = random.choice(pool[pid2])
        rows.append((pid1, a, pid2, b, 0))
    return pd.DataFrame(rows, columns=["id1", "key1", "id2", "key2", "label"])


pairs_same = sample_same_pairs(test_pool, N_SAME)
pairs_diff = sample_diff_pairs(test_pool, N_DIFF)
test_pairs = pd.concat([pairs_same, pairs_diff], ignore_index=True)

print("Test pairs:", test_pairs.shape[0])
print(test_pairs.head())

Train identities: 800
Test identities: 400
Train images used: 7999
Test ids with >=1 key: 400
Test pairs: 1000
       id1    key1      id2    key2  label
0  0691547  241782  0691547  241787      1
1  2425775  425773  2425775  425776      1
2  1122614  310278  1122614  310281      1
3  1715189  373294  1715189  373293      1
4  2325018  418232  2325018  418227      1


In [11]:
# Section 6: Decode image from RecordIO -> gray+resize -> HOG
def read_image_rgb_from_record(key: int) -> np.ndarray:
    s = rec.read_idx(int(key))
    if s is None:
        raise ValueError(f"Cannot read key={key} from RecordIO.")
    header, img_bytes = recordio.unpack(s)
    img = mx.image.imdecode(img_bytes).asnumpy()  # HxWx3 uint8
    return img

def load_gray_resized_from_key(key: int, out_hw=(H, W)) -> np.ndarray:
    img = read_image_rgb_from_record(key)
    if img.ndim == 3:
        g = rgb2gray(img)  # float
    else:
        g = img.astype(np.float32) / 255.0
    g = resize(g, out_hw, anti_aliasing=True, preserve_range=False).astype(np.float32)
    return g

def hog_feat(img_gray: np.ndarray) -> np.ndarray:
    return hog(
        img_gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        transform_sqrt=True,
        feature_vector=True,
    ).astype(np.float32)

def extract_hog_matrix_from_keys(keys: list[int], desc: str):
    X = []
    ok_keys = []
    for k in tqdm(keys, desc=desc, total=len(keys), leave=True):
        try:
            g = load_gray_resized_from_key(k)
            X.append(hog_feat(g))
            ok_keys.append(int(k))
        except Exception:
            continue
    if not X:
        raise RuntimeError("No images successfully processed for HOG.")
    return np.vstack(X), ok_keys


# ---- TRAIN HOG cache
hog_train_cache = CACHE_DIR / "hog_train.npy"
keys_train_cache = CACHE_DIR / "hog_train_keys.npy"

def build_train_hog():
    X, ok = extract_hog_matrix_from_keys(train_keys, "Extracting HOG (train)")
    np.save(hog_train_cache, X)
    np.save(keys_train_cache, np.array(ok, dtype=np.int64))
    return X, ok

if hog_train_cache.exists() and keys_train_cache.exists():
    saved = np.load(keys_train_cache).tolist()
    if saved == train_keys:
        X_train_hog = np.load(hog_train_cache)
        ok_train_keys = train_keys
        print("[OK] Loaded cached train HOG:", X_train_hog.shape)
    else:
        print("[WARN] Train keys changed; rebuilding train cache.")
        X_train_hog, ok_train_keys = build_train_hog()
else:
    X_train_hog, ok_train_keys = build_train_hog()

# align labels to ok_train_keys
key_to_label = dict(zip(train_keys, train_labels))
train_labels_ok = [key_to_label[k] for k in ok_train_keys]

print("Train HOG matrix:", X_train_hog.shape)

Extracting HOG (train): 100%|██████████| 7999/7999 [01:17<00:00, 103.26it/s]


Train HOG matrix: (7879, 8100)


In [12]:
# Section 6.1: PCA fit on train

pca = PCA(n_components=PCA_DIM, svd_solver="randomized", whiten=False, random_state=seed)
X_train_pca = pca.fit_transform(X_train_hog).astype(np.float32)
print("Train PCA matrix:", X_train_pca.shape)
print("Explained var ratio sum:", float(pca.explained_variance_ratio_.sum()))

train_df = pd.DataFrame({"key": ok_train_keys, "id": train_labels_ok})
train_df["x_50d"] = list(X_train_pca)
print(train_df.head())

Train PCA matrix: (7879, 50)
Explained var ratio sum: 0.3393559157848358
      key       id                                              x_50d
0  100223  0005445  [0.4270711, 2.4377184, 0.116854936, 2.219184, ...
1  100243  0005445  [0.016108718, -3.504635, 1.7435355, 1.5255786,...
2  100230  0005445  [-2.840978, 2.6794224, 0.59354115, 0.60512954,...
3  100253  0005445  [1.4747183, -0.5427148, 0.28210187, 1.0328326,...
4  100240  0005445  [-0.22140981, -0.23789483, -0.43706852, 2.2903...


In [13]:
# Section 6.2: Transform test pairs (unique keys)

all_test_keys = pd.unique(pd.concat([test_pairs["key1"], test_pairs["key2"]], ignore_index=True)).tolist()
all_test_keys = [int(k) for k in all_test_keys]

hog_test_cache = CACHE_DIR / "hog_test_unique.npy"
keys_test_cache = CACHE_DIR / "hog_test_unique_keys.npy"

def build_test_hog():
    X, ok = extract_hog_matrix_from_keys(all_test_keys, "Extracting HOG (test unique)")
    np.save(hog_test_cache, X)
    np.save(keys_test_cache, np.array(ok, dtype=np.int64))
    return X, ok

if hog_test_cache.exists() and keys_test_cache.exists():
    saved = np.load(keys_test_cache).tolist()
    if saved == all_test_keys:
        X_test_hog_unique = np.load(hog_test_cache)
        ok_test_keys = all_test_keys
        print("[OK] Loaded cached test HOG:", X_test_hog_unique.shape)
    else:
        print("[WARN] Test keys changed; rebuilding test cache.")
        X_test_hog_unique, ok_test_keys = build_test_hog()
else:
    X_test_hog_unique, ok_test_keys = build_test_hog()

key_to_row = {k: i for i, k in enumerate(ok_test_keys)}
X_test_pca_unique = pca.transform(X_test_hog_unique).astype(np.float32)

ok_set = set(key_to_row.keys())
mask_ok = test_pairs["key1"].isin(ok_set) & test_pairs["key2"].isin(ok_set)
test_pairs_ok = test_pairs[mask_ok].copy()

def get_pca_vec(key: int) -> np.ndarray:
    return X_test_pca_unique[key_to_row[int(key)]]

test_pairs_ok["x1_50d"] = test_pairs_ok["key1"].apply(get_pca_vec)
test_pairs_ok["x2_50d"] = test_pairs_ok["key2"].apply(get_pca_vec)

print("Test pairs kept:", test_pairs_ok.shape[0])
print(test_pairs_ok.head())

Extracting HOG (test unique): 100%|██████████| 1547/1547 [00:15<00:00, 101.61it/s]


Test pairs kept: 981
       id1    key1      id2    key2  label  \
0  0691547  241782  0691547  241787      1   
1  2425775  425773  2425775  425776      1   
2  1122614  310278  1122614  310281      1   
3  1715189  373294  1715189  373293      1   
4  2325018  418232  2325018  418227      1   

                                              x1_50d  \
0  [1.6500769, -1.0423802, -2.5139933, 1.9453683,...   
1  [-3.8617835, -1.1514735, 0.41975832, -0.110348...   
2  [1.2316856, -0.25339034, 0.80037856, -0.654154...   
3  [-0.8261317, -3.5472417, 0.9648387, 0.44891346...   
4  [-0.27732372, 2.8373485, 1.0108297, 1.3533628,...   

                                              x2_50d  
0  [2.2500753, 0.51610655, -1.1498715, 1.9546216,...  
1  [0.6462877, 1.453335, 0.90378624, -0.5386007, ...  
2  [2.540711, 1.3916947, -0.011684507, 0.331797, ...  
3  [-2.3503048, -2.7911465, 1.4248669, 0.26116124...  
4  [2.5869951, -1.1751128, -0.4001791, -2.1974018...  


In [14]:
# Section 7: Diagnostics (A1/A2/B1/B2/B3)

# A1: marginal normality per PCA dim
X = np.stack(train_df["x_50d"].to_numpy())
stats = []
for j in range(X.shape[1]):
    xj = X[:, j]
    stats.append({
        "dim": j,
        "skew": float(skew(xj)),
        "kurtosis_excess": float(kurtosis(xj, fisher=True)),
        "normaltest_p": float(normaltest(xj).pvalue),
    })
stats_df = pd.DataFrame(stats).sort_values("normaltest_p")
print(stats_df.head(10))

# A2: L2 distance distributions
x1 = np.stack(test_pairs_ok["x1_50d"].to_numpy())
x2 = np.stack(test_pairs_ok["x2_50d"].to_numpy())
d = norm(x1 - x2, axis=1)
test_pairs_ok["l2dist"] = d
print(test_pairs_ok.groupby("label")["l2dist"].describe())

# B1: scatter decomposition
train_df2 = train_df.copy()
train_df2["id"] = train_df2["id"].astype(str)

X = np.stack(train_df2["x_50d"].to_numpy()).astype(np.float64)
mu = X.mean(axis=0, keepdims=True)

S_W = np.zeros((X.shape[1], X.shape[1]), dtype=np.float64)
S_B = np.zeros_like(S_W)

for pid, g in train_df2.groupby("id"):
    Xg = np.stack(g["x_50d"].to_numpy()).astype(np.float64)
    ng = Xg.shape[0]
    mug = Xg.mean(axis=0, keepdims=True)
    Xc = Xg - mug
    S_W += Xc.T @ Xc
    dm = mug - mu
    S_B += ng * (dm.T @ dm)

S_T = (X - mu).T @ (X - mu)
print("||S_T - (S_W+S_B)||_F =", float(np.linalg.norm(S_T - (S_W + S_B))))

ratio_between = np.trace(S_B) / np.trace(S_T)
ratio_within = np.trace(S_W) / np.trace(S_T)
print("trace ratio between:", float(ratio_between), "within:", float(ratio_within))

# B2: within-class covariance heterogeneity
within_var = []
for pid, g in train_df2.groupby("id"):
    Xg = np.stack(g["x_50d"].to_numpy())
    if Xg.shape[0] < 2:
        continue
    covg = np.cov(Xg, rowvar=False)
    within_var.append({
        "id": pid,
        "n": int(Xg.shape[0]),
        "trace_cov": float(np.trace(covg)),
        "mean_var": float(np.mean(np.diag(covg))),
    })

within_df = pd.DataFrame(within_var)
print(within_df[["trace_cov","mean_var"]].describe(percentiles=[.05,.25,.5,.75,.95]))

# B3: whitening / Mahalanobis^2
X = np.stack(train_df2["x_50d"].to_numpy()).astype(np.float64)
Xc = X - X.mean(axis=0, keepdims=True)
Sigma = np.cov(Xc, rowvar=False)

ridge = 1e-5
Sigma_r = Sigma + ridge*np.eye(Sigma.shape[0])

c, low = cho_factor(Sigma_r, lower=True, check_finite=False)
r2 = np.einsum("ij,ij->i", Xc, cho_solve((c, low), Xc.T, check_finite=False).T)
print("mean r2:", float(r2.mean()), "theory:", Sigma.shape[0])
print("var  r2:", float(r2.var()),  "theory:", 2*Sigma.shape[0])


    dim      skew  kurtosis_excess   normaltest_p
1     1 -0.023046        -0.895946  4.210444e-230
0     0 -0.272330        -0.758070  9.173688e-140
10   10 -0.229980         0.594853   6.157086e-31
24   24 -0.169170         0.397323   7.797447e-17
27   27  0.196990         0.307186   8.527871e-17
14   14  0.109965         0.454164   3.335085e-14
8     8 -0.199423         0.042062   5.264893e-12
26   26 -0.015262         0.440521   2.410209e-10
32   32 -0.079285         0.325127   3.053839e-08
4     4 -0.134327        -0.130892   3.406140e-07
       count      mean       std       min       25%       50%       75%  \
label                                                                      
0      490.0  7.493417  1.180618  4.295965  6.691135  7.503825  8.217067   
1      491.0  7.135336  1.334853  3.536125  6.228660  7.227926  8.112872   

             max  
label             
0      10.747455  
1      11.099530  
||S_T - (S_W+S_B)||_F = 5.296471738873538e-11
trace ratio between: 0.